#  LABORATORIO 001

# **Sistema anti-fraude**

Se genera un conjunto simulado de 300 transacciones (con más probabilidad de fraude en horario
de madrugada y en montos altos) y se visualiza, para observar el patrón antes de aplicar cualquier
algoritmo (lo que se hará desde la Semana 5):

## <span style="color:#2B5B84;">1. Importación del Ecosistema de Datos</span>

Para el desarrollo del laboratorio, inicializamos las librerías base para la manipulación numérica, análisis estructurado y renderizado gráfico:

| Librería | Alias | Propósito Académico |
| :--- | :---: | :--- |
| **NumPy** | `np` | Manejo de arreglos multidimensionales y generación de variables aleatorias. |
| **Pandas** | `pd` | Estructuración de datos en DataFrames y manipulación tabular. |
| **Matplotlib** | `plt` | Construcción, renderizado y personalización visual de gráficos. |

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## <span style="color:#2B5B84;">2. Generación de Datos Sintéticos</span>

Configuramos el entorno estocástico definiendo la semilla de reproducibilidad y modelando las variables principales de las transacciones:

* **Semilla (`seed = 42`)**: Garantiza la **reproducibilidad** de los experimentos.
* **Muestra ($n = 300$)**: Volumen total de transacciones simuladas.

| Variable | Tipo de Dato | Distribución / Algoritmo | Descripción |
| :--- | :---: | :--- | :--- |
| `hora` | Entero | Uniforme Discreta $\sim U(0, 23)$ | Representa la hora del día en formato 24 horas. |
| `monto` | Flotante | Exponencial $\sim \text{Exp}(\lambda=150) + 10$ | Simula compras reales: alta frecuencia de montos bajos y baja frecuencia de montos altos (mínimo S/ 10). |

In [4]:
np.random.seed(42)
n = 300
hora = np.random.randint(0, 24, n)
monto = np.random.exponential(scale=150, size=n) + 10

## <span style="color:#2B5B84;">3. Modelado de Probabilidad de Fraude</span>

Establecemos la probabilidad de fraude $P(\text{Fraude})$ combinando reglas de negocio basadas en el comportamiento operativo:

$$\text{P(Fraude)} = \text{Base}(3\%) + \text{Riesgo Horario}(35\%) + \text{Riesgo Monto}(25\%)$$

<blockquote style="background-color: #f9f9f9; border-left: 4px solid #d03b3b; padding: 8px;">
<b>Reglas de Evaluación:</b><br>
1. <b>Horario Crítico:</b> Incrementa +0.35 de probabilidad si ocurre entre las 23:00 h y las 05:00 h.<br>
2. <b>Monto Atípico:</b> Incrementa +0.25 de probabilidad si el monto supera los S/ 400.<br>
3. <b>Truncamiento (Clip):</b> Normaliza la probabilidad en el rango $[0.0, 0.9]$.
</blockquote>

In [5]:
# Fraude mas probable de madrugada (0-5h, 23h) y en montos altos
prob_fraude = 0.03 + 0.35 * ((hora <= 5) | (hora >= 23)) + 0.25 * (monto > 400)
prob_fraude = np.clip(prob_fraude, 0, 0.9)
es_fraude = (np.random.rand(n) < prob_fraude).astype(int)